# 🔐 Notebook 2: CRC32 vs MD5 vs SHA-256 vs BLAKE2b

In Notebook 1 we saw silent corruption. Now we fix it by storing a **checksum** next to the data and recomputing it on read.

Not all checksums are the same. They trade **speed** against **strength**:

| Algorithm | Size | Designed for | Collision-resistant vs attacker? |
|---|---|---|---|
| **CRC32** | 4 bytes | accidental bit errors (disk, Ethernet frame) | ❌ no |
| **MD5** | 16 bytes | fast content fingerprint | ❌ broken since 2004 |
| **SHA-256** | 32 bytes | cryptographic integrity, signatures | ✅ yes |
| **BLAKE2b** | 32 bytes (tunable) | modern SHA-2 alternative | ✅ yes, and faster |

Rule of thumb:
- You only worry about **random bit flips** → CRC32 is plenty and basically free.
- You need to **detect tampering** by someone malicious → a plain hash stored next to the data
  is **not enough, whichever algorithm you pick**. You need a *secret* — an HMAC (Notebook 3).

> ⚠️ Two different properties get conflated constantly, so let's separate them now and then
> demonstrate both below:
>
> - **Collision resistance** is a property of the *algorithm*. CRC32 has none — we will find a
>   collision in this notebook, in a fraction of a second. SHA-256 has it.
> - **Tamper resistance** is a property of the *scheme*. Storing `digest(data)` next to `data`
>   has none, no matter how strong the digest, because an attacker who can rewrite one can
>   rewrite the other. Only a key changes that.
>
> A checksum answers *"did these bytes change by accident?"*. It does not answer *"did someone
> change these bytes on purpose?"* — that is a MAC, and it is a different tool.

Big real-world storage systems use *both*: CRC32 per disk page to catch hardware faults cheaply, plus a cryptographic hash at the object level for end-to-end integrity.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/checksum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Everything here uses only the Python standard library (`zlib`, `hashlib`). No installs required beyond `uv sync`.

## 🛠️ Helpers: store `[len(digest)][digest][payload]` on disk

A very common on-disk layout for a checksummed record:

```
┌──────────┬───────────────────┬──────────────────────────────┐
│ len  (2B)│ digest (N bytes)  │  payload (the actual bytes)  │
└──────────┴───────────────────┴──────────────────────────────┘
```

Real systems (Kafka, SQLite, Postgres pages, Parquet, ...) use variations of this idea — checksum + length + payload.

In [ ]:
import zlib, hashlib, time, os, tempfile

def crc32(b):    return zlib.crc32(b).to_bytes(4, 'big')
def md5(b):      return hashlib.md5(b).digest()
def sha256(b):   return hashlib.sha256(b).digest()
def blake2b(b):  return hashlib.blake2b(b, digest_size=32).digest()

ALGS = {'crc32': crc32, 'md5': md5, 'sha256': sha256, 'blake2b': blake2b}

def store(path, payload, alg):
    digest = ALGS[alg](payload)
    with open(path, 'wb') as f:
        f.write(len(digest).to_bytes(2, 'big'))
        f.write(digest)
        f.write(payload)

def load(path, alg):
    with open(path, 'rb') as f:
        n = int.from_bytes(f.read(2), 'big')
        stored_digest = f.read(n)
        payload = f.read()
    if ALGS[alg](payload) != stored_digest:
        raise ValueError(f'checksum mismatch ({alg}) — file is corrupt')
    return payload


## ✅ Round-trip + corruption test

For each algorithm:
1. Store some bytes with the checksum.
2. Load them back — should succeed.
3. Flip a single byte in the file, try again — should be detected.

In [ ]:
WORKDIR = tempfile.mkdtemp(prefix='chk_')
data = b'hello world ' * 100

for alg in ALGS:
    path = os.path.join(WORKDIR, f'data.{alg}')
    store(path, data, alg)
    ok = load(path, alg) == data
    print(f'{alg:8s} round-trip ok? {ok}')

    # Corrupt one byte in the payload and try to read again.
    raw = bytearray(open(path, 'rb').read())
    raw[-1] ^= 0xFF
    open(path, 'wb').write(raw)

    try:
        load(path, alg)
    except ValueError as e:
        print(f'         ✅ corruption detected: {e}')
    else:
        raise AssertionError(f'{alg} failed to detect a flipped byte')
    assert ok, f'{alg} round-trip failed'


## 🔓 What a checksum does **not** buy you

Every algorithm above caught a random byte flip. Now change the threat model: instead of a
cosmic ray, assume someone who can write to the file. They will simply recompute the
checksum.

In [ ]:
tamper_path = os.path.join(WORKDIR, 'ledger.bin')
record = b'transfer to=mallory amount=10'

for alg in ALGS:
    store(tamper_path, record, alg)

    # Attacker rewrites the payload AND the digest — they have the same tools we do.
    forged = b'transfer to=mallory amount=99999'
    store(tamper_path, forged, alg)          # exactly what `store` does; no secret needed

    accepted = load(tamper_path, alg)
    print(f'{alg:8s} tampered record accepted? {accepted == forged}')
    assert accepted == forged, 'the forgery should sail straight through'

print('\n💥 4/4, including SHA-256 and BLAKE2b. The algorithm was never the weak part —')
print('   the scheme is. `digest(data)` stored beside `data` is an integrity check against')
print('   ACCIDENTS. Against an adversary it is decoration. Notebook 3 fixes this with HMAC.')

### And a weakness that *is* specific to CRC32

The scheme problem above hits every algorithm equally. This one does not: CRC32 is 32 bits
wide, so by the birthday bound you expect a collision after about `2^16 ≈ 65,000` messages.
That is not a theoretical concern — it is a fraction of a second of laptop time.

Two *different* payloads with the *same* CRC32 means a corruption that CRC32 cannot see, and
a "content address" that maps two documents to one name.

In [ ]:
import random as _random

def find_crc32_collision(prefix=b'transfer amount=100 nonce=', seed=7, limit=1_000_000):
    """Birthday search for two distinct messages sharing a CRC32."""
    rng = _random.Random(seed)
    seen = {}
    for tries in range(1, limit + 1):
        m = prefix + f'{rng.getrandbits(64):016x}'.encode()
        c = zlib.crc32(m)
        if c in seen and seen[c] != m:
            return seen[c], m, c, tries
        seen[c] = m
    raise RuntimeError('no collision found')

t0 = time.perf_counter()
m1, m2, crc, tries = find_crc32_collision()
elapsed = time.perf_counter() - t0

print(f'found in {tries:,} tries, {elapsed * 1000:.0f} ms')
print(f'  {m1.decode()}')
print(f'  {m2.decode()}')
print(f'  both crc32 = {crc:#010x}')

assert m1 != m2 and zlib.crc32(m1) == zlib.crc32(m2)
assert hashlib.sha256(m1).digest() != hashlib.sha256(m2).digest()   # SHA-256 tells them apart
print(f'\n✔ two different messages, one CRC32, found on a laptop in {elapsed * 1000:.0f} ms.')
print('  The same search against SHA-256 would take ~2^128 tries.')

### So what *is* CRC32 good for?

Quite a lot — as long as you use it for the job it was designed for. CRC32 is not a weak hash;
it is a **strong error-detecting code**. Its guarantees are combinatorial, not probabilistic:

- **Any single-bit error** — always detected.
- **Any burst error up to 32 bits** — always detected. (This is why the collision search above
  needed *random* nonces: a difference confined to a few adjacent bytes can never be divisible
  by the degree-32 generator polynomial, so nearby messages never collide.)
- **Any odd number of bit flips** — always detected.

That maps exactly onto how disks and network links actually fail, which is why Ethernet frames,
Postgres pages, and Kafka record batches all use CRC32/CRC32C. It is the right tool for
hardware faults and the wrong tool for adversaries — and those are different problems, not
different strengths of the same problem.

In [ ]:
# Verify the burst-error guarantee: flip every possible run of up to 4 bytes and
# confirm CRC32 catches all of them. (A 32-bit burst is 4 contiguous bytes.)
base = b'account=alice; balance=100; ts=1700000000'
base_crc = zlib.crc32(base)
undetected = 0
checked = 0
for start in range(len(base)):
    for width in (1, 2, 3, 4):
        if start + width > len(base):
            continue
        corrupt = bytearray(base)
        for j in range(start, start + width):
            corrupt[j] ^= 0xFF          # worst case: every bit in the burst flipped
        checked += 1
        if zlib.crc32(bytes(corrupt)) == base_crc:
            undetected += 1

print(f'checked {checked} burst errors of 1-4 bytes: {undetected} undetected')
assert undetected == 0, 'CRC32 must catch every burst up to 32 bits'
print('✔ CRC32 caught every one — that is a guarantee, not a probability')

## 🌊 Streaming: checksumming files you can't fit in RAM

Everything above reads the whole payload into memory. For a 50 GB backup file or an HTTP upload that would be a disaster. The trick: **feed the data to the hasher in chunks** — every hash in Python supports `.update(chunk)` and gives you the same digest at the end.

This is how `s3 cp`, `rsync`, `sha256sum`, and Git compute hashes for huge files.

In [ ]:
CHUNK = 64 * 1024  # 64 KB at a time

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(CHUNK)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# Make a 20 MB file of random bytes
big = os.path.join(WORKDIR, 'big.bin')
with open(big, 'wb') as f:
    f.write(os.urandom(20 * 1024 * 1024))

print('streaming sha256:', sha256_file(big)[:16], '...')
# Sanity check — same result as hashing the whole thing at once
whole = hashlib.sha256(open(big, 'rb').read()).hexdigest()
print('whole-file sha256:', whole[:16], '...')
print('match?', sha256_file(big) == whole)


## ⚡ Performance comparison on 20 MB

Numbers will vary by CPU. What matters is the **shape**: CRC32 is typically the cheapest, MD5 the cheapest cryptographic-ish option, SHA-256 stronger but slower, BLAKE2b usually beats SHA-256 in pure software.

In [ ]:
blob = os.urandom(20 * 1024 * 1024)

print(f'{"algorithm":10s} {"ms/20MB":>10s} {"MB/s":>10s}')
print('-' * 32)
for name, fn in ALGS.items():
    runs = 5
    t0 = time.perf_counter()
    for _ in range(runs):
        fn(blob)
    ms = (time.perf_counter() - t0) * 1000 / runs
    mb_s = 20 / (ms / 1000)
    print(f'{name:10s} {ms:10.2f} {mb_s:10.0f}')


## 📊 Choosing: real-world examples

| System | What they use | Why |
|---|---|---|
| **TCP / IP header** | 16-bit ones'-complement checksum | catches common bit flips cheaply in hardware |
| **Ethernet frames** | CRC32 | hardware-cheap, catches all 1-3 bit burst errors |
| **ZFS / Btrfs** | Fletcher / CRC32C + optional SHA-256 | per-block bit-rot detection + scrub |
| **Postgres pages** | CRC32C (when `data_checksums` is on) | detect torn/corrupt 8 KB pages |
| **Kafka records** | CRC32C per record batch | reject corrupt batches on disk / on wire |
| **Git** | SHA-1 (→ SHA-256) | content-addressed storage; objects named by their hash. Note Git's hash is *not* a tamper check on its own — signed commits/tags add the key. |
| **S3 ETag** | MD5 of object (or composite for multipart) | client can verify download |
| **HTTPS / TLS** | HMAC-SHA256 or AEAD (GCM) | integrity **and** authenticity of every packet |
| **Signed URLs / JWT** | HMAC-SHA256 | server signs, nobody else can forge |

Most production storage layers combine **cheap per-block CRC** (for hardware faults) with a
**cryptographic hash at the object level** (for end-to-end integrity and dedup).

### Picking one: three questions

1. **Who might change the bytes?** Hardware or a buggy network → CRC32 is plenty, and free.
   A person → no unkeyed checksum helps; go to Notebook 3.
2. **How many things am I naming?** A 32-bit checksum collides after ~65k items. If the digest
   is an *identifier* (content addressing, dedup, cache keys), you need 256 bits.
3. **Where does it get verified?** A checksum stored and checked by the same component only
   catches that component's faults. End-to-end integrity means the *writer* computes it and the
   *reader* verifies it, with everything in between untrusted.

👉 Notebook 3 shows the two patterns you need on top of hashes: **ETag** (for HTTP conditional requests) and **HMAC** (for tamper detection with a secret key).